# 00 ? Environment Check

Setup checks only. No competition data, models, pretrained weights or training are used.

## 1. Python and operating system

The selected kernel must use the project-local `.venv`.

In [1]:
import platform
import sys
from pathlib import Path

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Operating system:", platform.platform())

Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Executable: C:\Users\leoca\Kaggle-Competition\.venv\Scripts\python.exe
Operating system: Windows-11-10.0.26200-SP0


## 2. Locate the project and verify paths

Run from the repository root or `notebooks/`. Structural markers avoid relying on the repository folder name.

In [2]:
def find_project_root(start):
    """Find this project using its requirements and numbered notebook."""
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks" / "00_Environment_Check.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this project root or its notebooks directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
assert find_project_root(PROJECT_ROOT) == PROJECT_ROOT
assert find_project_root(PROJECT_ROOT / "notebooks") == PROJECT_ROOT

# Check the active interpreter, not merely the name shown in the kernel picker.
expected_environment = (PROJECT_ROOT / ".venv").resolve()
assert Path(sys.prefix).resolve() == expected_environment, "Select the project .venv kernel."
assert Path(sys.executable).resolve().parent == expected_environment / "Scripts"
assert sys.prefix != sys.base_prefix, "A virtual environment is required."
assert sys.version_info[:3] == (3, 14, 3), "This setup was verified with Python 3.14.3."
print("Project root:", PROJECT_ROOT)
print("Project .venv: verified")

# Existence checks only: do not inspect or load any dataset files.
for relative_path in ("data/raw", "data/processed", "outputs/figures", "outputs/metrics", "outputs/models"):
    directory = PROJECT_ROOT / relative_path
    assert directory.is_dir(), f"Missing directory: {directory}"
    print(relative_path, "->", directory, "[OK]")

Project root: C:\Users\leoca\Kaggle-Competition
Project .venv: verified
data/raw -> C:\Users\leoca\Kaggle-Competition\data\raw [OK]
data/processed -> C:\Users\leoca\Kaggle-Competition\data\processed [OK]
outputs/figures -> C:\Users\leoca\Kaggle-Competition\outputs\figures [OK]
outputs/metrics -> C:\Users\leoca\Kaggle-Competition\outputs\metrics [OK]
outputs/models -> C:\Users\leoca\Kaggle-Competition\outputs\models [OK]


## 3. Dependency versions

Import each requested dependency and report its installed distribution version.

In [3]:
from importlib import import_module, metadata

DEPENDENCIES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "Pillow": "PIL",
    "jupyterlab": "jupyterlab",
    "ipykernel": "ipykernel",
    "nbformat": "nbformat",
}
for distribution, module in DEPENDENCIES.items():
    import_module(module)
    print(f"{distribution}: {metadata.version(distribution)}")

torch: 2.14.1+cpu


torchvision: 0.29.1+cpu
numpy: 2.5.2


pandas: 3.0.6
matplotlib: 3.11.2


scikit-learn: 1.9.1
Pillow: 12.3.0


jupyterlab: 4.6.4

ipykernel: 7.4.0

nbformat: 5.11.1

## 4. PyTorch CPU check

A small tensor operation checks basic execution. CPU-only wheels are expected: `torch.cuda.is_available()` should be `False`. This does not prove the computer has no GPU; GPU training will be assessed separately.

In [4]:
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

# Create a tiny CPU tensor; no model or dataset is involved.
values = torch.tensor([1.0, 2.0, 3.0, 4.0], device="cpu")
total = values.sum()
assert total.item() == 10.0
assert torch.version.cuda is None, "Expected a CPU-only PyTorch wheel."
assert not torch.cuda.is_available(), "Expected CPU-only operation for this setup."
print("Tensor device:", values.device)
print("Tensor sum:", total.item())
print("All environment checks passed.")

torch:

2.14.1+cpu

torchvision:

0.29.1+cpu

CUDA build:

...

Tensor device:

cpu

Tensor sum:

10.0

All environment checks passed.